#### Шаг 1. Запуск Qdrant через Docker

Минимальный запуск **(данные в памяти)**:

```
docker run -p 6333:6333 qdrant/qdrant
```

                  
* Порт **6333** — HTTP API,
* Данные **исчезнут после остановки контейнера.**

Для **продакшена** — добавьте volume (сохранение данных на диск):

```
docker run -p 6333:6333 \
  -v $(pwd)/qdrant_storage:/qdrant/storage \
  qdrant/qdrant
```

#### Шаг 2. Установка зависимостей

* qdrant-client — официальный Python-клиент,
* sentence-transformers — для генерации эмбеддингов.

In [1]:
pip install qdrant-client sentence-transformers -q

Note: you may need to restart the kernel to use updated packages.


#### Полный рабочий пример: qdrant_rag_demo.py

In [2]:
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer
from qdrant_client.models import VectorParams, Distance, PointStruct

# === 1. Данные ===
documents = [
    "FastAPI — это современный фреймворк для построения API на Python.",
    "Alembic используется для управления миграциями базы данных в SQLAlchemy.",
    "Docker упрощает развёртывание приложений через контейнеризацию."
]

# === 2. Embedding-модель ===
model = SentenceTransformer("all-MiniLM-L6-v2")  # 384 измерения
embeddings = model.encode(documents)

# === 3. Подключение к Qdrant ===
client = QdrantClient("localhost", port=6333)

# === 4. Удаление и создание коллекции ===
collection_name = "docs"
if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)

# === 5. Загрузка данных ===
points = [
    PointStruct(
        id=i,
        vector=embedding.tolist(),
        payload={"text": doc, "source": "internal_docs"}
    )
    for i, (doc, embedding) in enumerate(zip(documents, embeddings))
]

client.upload_points(collection_name=collection_name, points=points)

# === 6. Поиск ===
query = "Как обновить структуру базы данных?"
query_vector = model.encode(query).tolist()

search_results = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=1
)

# === 7. Вывод результата ===
print(search_results.points[0].payload["text"])

/opt/anaconda3/envs/LLM_course/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 20905.56it/s]


Alembic используется для управления миграциями базы данных в SQLAlchemy.


#### Как проверить, что данные загружены?

In [3]:
all_points, _ = client.scroll(collection_name="docs", limit=10)
for point in all_points:
    print(f"ID: {point.id}\nText: {point.payload['text']}\n---")

ID: 0
Text: FastAPI — это современный фреймворк для построения API на Python.
---
ID: 1
Text: Alembic используется для управления миграциями базы данных в SQLAlchemy.
---
ID: 2
Text: Docker упрощает развёртывание приложений через контейнеризацию.
---


#### Базовый семантический поиск

In [4]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
from sentence_transformers import SentenceTransformer

docs = [
    "FastAPI поддерживает автоматическую генерацию OpenAPI-документации.",
    "Alembic позволяет создавать и применять миграции базы данных.",
    "Docker упрощает развёртывание через контейнеризацию."
]

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(docs)

client = QdrantClient("localhost", port=6333)
if client.collection_exists("task1"):
    client.delete_collection("task1")

client.create_collection(
    collection_name="task1",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)

points = [PointStruct(id=i, vector=emb.tolist(), payload={"text": d}) for i, (d, emb) in enumerate(zip(docs, embeddings))]
client.upload_points("task1", points)

query = "Как обновлять структуру БД?"
query_vec = model.encode(query).tolist()
result = client.query_points("task1", query=query_vec, limit=1)
print(result.points[0].payload["text"])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7341.92it/s]


Alembic позволяет создавать и применять миграции базы данных.


#### Поиск с фильтрацией по source

In [5]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct, Filter, FieldCondition, MatchValue
from sentence_transformers import SentenceTransformer

docs = [
    {"text": "FastAPI — фреймворк для API на языке Python.", "source": "docs"},
    {"text": "Ошибка 500 в логах.", "source": "logs"},
    {"text": "Alembic управляет миграциями базы данных в Python-проектах.", "source": "docs"}
]

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode([d["text"] for d in docs])

client = QdrantClient("localhost", port=6333)
if client.collection_exists("task2"):
    client.delete_collection("task2")

client.create_collection(
    collection_name="task2",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)

points = [
    PointStruct(id=i, vector=emb.tolist(), payload=d)
    for i, (d, emb) in enumerate(zip(docs, embeddings))
]
client.upload_points("task2", points)

query = "Фреймворк на Python для API"
query_vec = model.encode(query).tolist()

result = client.query_points(
    "task2",
    query=query_vec,
    query_filter=Filter(must=[FieldCondition(key="source", match=MatchValue(value="docs"))]),
    limit=1
)
print(result.points[0].payload["text"])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11000.82it/s]


FastAPI — фреймворк для API на языке Python.


#### Вывод payload целиком

In [6]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
from sentence_transformers import SentenceTransformer

docs = [
    {"text": "Docker", "category": "infra", "version": "20.10"},
    {"text": "Alembic", "category": "db", "version": "1.13"},
    {"text": "FastAPI", "category": "api", "version": "0.100"}
]

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode([d["text"] for d in docs])

client = QdrantClient("localhost", port=6333)
if client.collection_exists("task3"):
    client.delete_collection("task3")

client.create_collection(
    collection_name="task3",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)


points = [
    PointStruct(id=i, vector=emb.tolist(), payload=d)
    for i, (d, emb) in enumerate(zip(docs, embeddings))
]
client.upload_points("task3", points)

query = "Инструмент для миграций"
query_vec = model.encode(query).tolist()
result = client.query_points("task3", query=query_vec, limit=1)
print(result.points[0].payload["category"])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6003.02it/s]


db
